<a href="https://colab.research.google.com/github/luca-fce/siniestros-futbol-caba/blob/main/notebook/3_1_carga_y_exploracion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 3.1 Carga y exploración inicial

**Trabajo práctico — Laboratorio de Métodos Cuantitativos Aplicados a la Gestión (FCE-UBA)**

Pregunta del trabajo: *¿Se registra un aumento significativo de siniestros viales en la Ciudad de Buenos Aires durante las 12 horas posteriores a la derrota de los equipos más populares de fútbol (Selección Argentina, River Plate y Boca Juniors)?*

En este notebook se cargan las tres bases de datos **tal como vienen de la fuente**, sin modificarlas, y se describe su estructura y su calidad. Los problemas detectados se corrigen en el notebook **3.3**.

| Base | Fuente | Archivo en el repositorio |
|---|---|---|
| Siniestros viales CABA 2019–2025 (hechos y víctimas) | Observatorio de Movilidad y Seguridad Vial, GCBA (Buenos Aires Data) | `datos/Siniestros Viales 2019 - 2025/` |
| Partidos de River Plate y Boca Juniors 2021–2025 | FBref | `datos/Temporadas River - Boca/` |
| Resultados internacionales 1872–2026 | Kaggle (martj42) | `datos/Resultados Internacionales 1872 - 2026/` |

> **Uso de IA.** Todas las celdas de este notebook se elaboraron con asistencia de Claude (Anthropic). Prompt utilizado: «Cargá las bases de siniestros viales de CABA, de partidos de River y Boca (FBref) y de resultados internacionales; mostrá primeras y últimas filas, estructura, tipos de dato, cobertura temporal y geográfica, nulos, duplicados y estadísticas descriptivas, y comentá los problemas de calidad». El grupo revisó el código, verificó los resultados contra las fuentes y validó las interpretaciones.

Librerías y ruta del repositorio. Los datos se leen directamente desde GitHub, así el notebook corre en Colab sin subir archivos.

In [1]:
import pandas as pd
import numpy as np

REPO = 'https://raw.githubusercontent.com/luca-fce/siniestros-futbol-caba/main/'
pd.set_option('display.max_columns', 30)

## A. Siniestros viales

Son dos tablas que se relacionan por `id_siniestro`: **hechos** (un registro por siniestro) y **víctimas** (un registro por persona). Vienen separadas por `;` y con codificación UTF-8 con BOM. Se leen todas las columnas como texto para poder ver los valores tal como fueron cargados.

### a) Carga: primeras y últimas filas

In [2]:
RUTA_SIN = REPO + 'datos/Siniestros%20Viales%202019%20-%202025/'
hechos_raw = pd.read_csv(RUTA_SIN + 'siniestros_viales_hechos.csv', sep=';', encoding='utf-8-sig', dtype=str)
victimas_raw = pd.read_csv(RUTA_SIN + 'siniestros_viales_victimas.csv', sep=';', encoding='utf-8-sig', dtype=str)
hechos_raw.head()

,id_siniestro,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,hora_siniestro,rango_horario,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,gravedad_siniestro
0,LC-2019-0000053,1,1,0,0,2019-01-01,2019,1,1,01:55:00,1,NaN,Comuna 8,AUTOPISTA,POINT (101813.84712503915943671 95578.55507230...,-58.44351,-34.669125,AUTO-UTILITARIO,AUTO,UTILITARIO,LEVE
1,LC-2019-0000647,1,0,1,0,2019-01-01,2019,1,1,18:50:00,18,DE LOS CONSTITUYENTES AV. y HABANA,Comuna 12,AVENIDA,POINT (97510.27137648081406951 105087.83369881...,-58.490436,-34.583403,MOTO-AUTO,MOTO,AUTO,GRAVE
2,LC-2019-0000445,3,3,0,0,2019-01-01,2019,1,1,13:40:00,13,NaN,Comuna 11,NaN,POINT (94371.44237885093025398 102884.19365105...,-58.524662,-34.603255,AUTO-AUTO,AUTO,AUTO,LEVE
3,LC-2019-0000194,1,1,0,0,2019-01-01,2019,1,1,07:15:00,7,NaN,Comuna 9,NaN,POINT (94030.76669932194636203 97681.071761248...,-58.528413,-34.650156,AUTO-CAMION,AUTO,CAMION,LEVE
4,LC-2019-0000329,1,1,0,0,2019-01-01,2019,1,1,12:24:00,12,NaN,Comuna 4,NaN,POINT (105367.86665769023238681 97877.75085908...,-58.404748,-34.648387,AUTO-MOVIL,AUTO,MOVIL,LEVE


Últimas filas de la tabla de hechos:

In [3]:
hechos_raw.tail()

,id_siniestro,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,fecha_siniestro,anio_siniestro,mes_siniestro,dia_siniestro,hora_siniestro,rango_horario,direccion_normalizada_siniestro,comuna_siniestro,tipo_de_via_siniestro,geocodificacion_plana,longitud_siniestro,latitud_siniestro,participantes_siniestro,modo_desplazamiento_victima,contraparte_siniestro,gravedad_siniestro
65813,LC-2025-0720215,1,1,0,0,2025-9-9,2025,9,9,08:00:00,8,"BRAVO, MARIO 556",5,CALLE,POINT (104277.730887 102927.628643),-58.416664,-34.602870,MOTO-CAMION,MOTO,CAMION,LEVE
65814,LC-2025-0719795,SD,1,0,0,2025-9-9,2025,9,9,07:59:00,7,MORENO 3400,5,CALLE,POINT (104483.433519 101657.976209),-58.414415,-34.614314,SD-AUTO,SD,AUTO,LEVE
65815,LC-2025-0720513,SD,1,0,0,2025-9-9,2025,9,9,17:26:00,17,JUNCAL 2800,2,CALLE,POINT (105332.105712 104466.328894),-58.405179,-34.588995,MOTO-AUTO,MOTO,AUTO,LEVE
65816,LC-2025-0720230,1,1,0,0,2025-9-9,2025,9,9,07:40:00,7,LARREA 800,2,CALLE,POINT (105575.720563 103332.553006),-58.402516,-34.599214,BICICLETA-AUTO,BICICLETA,AUTO,LEVE
65817,LC-2025-0720576,SD,1,0,0,2025-9-9,2025,9,9,17:57:00,17,"CEVALLOS, VIRREY 1400",1,CALLE,POINT (106872.781288 100488.5469),-58.388353,-34.624843,MOTO-AUTO,MOTO,AUTO,LEVE


Primeras y últimas filas de la tabla de víctimas:

In [4]:
pd.concat([victimas_raw.head(3), victimas_raw.tail(5)])

,id_siniestro,fecha_siniestro,anio_siniestro,modo_desplazamiento_victima,sexo_victima,edad_victima,GRAVEdad_victima,rol_victima,fecha_fallecimiento_victima
0,LC-2019-0022650,2019-01-11,2019,BICICLETA,F,25,GRAVE,CICLISTA,NaN
1,LC-2019-0068291,2019-02-01,2019,BICICLETA,F,16,GRAVE,CICLISTA,NaN
2,LC-2019-0139186,2019-03-06,2019,BICICLETA,F,32,GRAVE,CICLISTA,NaN
75192,LC-2025-0864021,2025-11-10,2025,UTILITARIO,SD,SD,LEVE,SD,NaN
75193,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75194,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75195,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75196,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### b) Estructura: dimensiones, columnas, tipos de dato y cobertura

In [5]:
print('Hechos:  ', hechos_raw.shape[0], 'filas x', hechos_raw.shape[1], 'columnas')
print('Víctimas:', victimas_raw.shape[0], 'filas x', victimas_raw.shape[1], 'columnas\n')
print(hechos_raw.dtypes)

Hechos:   65818 filas x 21 columnas
Víctimas: 75197 filas x 9 columnas

id_siniestro                        str
numero_total_de_victimas            str
numero_victimas_leve_siniestro      str
numero_victimas_grave_siniestro     str
numero_victimas_mortal_siniestro    str
fecha_siniestro                     str
anio_siniestro                      str
mes_siniestro                       str
dia_siniestro                       str
hora_siniestro                      str
rango_horario                       str
direccion_normalizada_siniestro     str
comuna_siniestro                    str
tipo_de_via_siniestro               str
geocodificacion_plana               str
longitud_siniestro                  str
latitud_siniestro                   str
participantes_siniestro             str
modo_desplazamiento_victima         str
contraparte_siniestro               str
gravedad_siniestro                  str
dtype: object


**Cobertura temporal:** cantidad de registros por año en cada tabla.

In [6]:
pd.DataFrame({'hechos': hechos_raw['anio_siniestro'].value_counts(),
              'victimas': victimas_raw['anio_siniestro'].value_counts()}).sort_index()

,hechos,victimas
anio_siniestro,,
2019,10155,11886
2020,6061,6936
2021,7818,9053
2022,9536,10647
2023,9927,11075
2024,10569,12481
2025,11752,13115


**Cobertura geográfica:** comuna del siniestro y rango de las coordenadas.

In [7]:
print(hechos_raw['comuna_siniestro'].value_counts(dropna=False).head(20))
coords = hechos_raw[['latitud_siniestro', 'longitud_siniestro']].apply(pd.to_numeric, errors='coerce')
coords.describe()

comuna_siniestro
Comuna 1     5790
Comuna 15    4361
Comuna 3     3994
Comuna 4     3720
Comuna 9     3653
Comuna 14    3648
Comuna 12    3623
Comuna 7     3537
Comuna 13    3249
Comuna 10    3133
NaN          3017
Comuna 11    2997
Comuna 5     2619
Comuna 8     2480
Comuna 6     2255
Comuna 2     1935
1            1400
3             861
15            852
14            848
Name: count, dtype: int64


,latitud_siniestro,longitud_siniestro
count,62787.000000,62787.000000
mean,-34.611524,-58.442240
std,0.031871,0.044195
min,-34.705250,-58.531218
25%,-34.632236,-58.476580
50%,-34.612594,-58.441760
75%,-34.590188,-58.405282
max,-34.534254,-58.351517


### c) Calidad: nulos, duplicados y estadísticas descriptivas

In [8]:
pd.DataFrame({'nulos_hechos': hechos_raw.isnull().sum()}).join(
    pd.DataFrame({'nulos_victimas': victimas_raw.isnull().sum()}), how='outer').fillna('-')

,nulos_hechos,nulos_victimas
GRAVEdad_victima,-,4.0
anio_siniestro,0.0,4.0
comuna_siniestro,3017.0,-
contraparte_siniestro,0.0,-
dia_siniestro,0.0,-
direccion_normalizada_siniestro,12900.0,-
edad_victima,-,4.0
fecha_fallecimiento_victima,-,74494.0
fecha_siniestro,0.0,4.0
geocodificacion_plana,2367.0,-


Duplicados: en hechos se revisa si se repite el identificador; en víctimas, si hay filas idénticas.

In [9]:
print('Hechos - filas duplicadas:', hechos_raw.duplicated().sum(), '| id_siniestro repetidos:', hechos_raw['id_siniestro'].duplicated().sum())
print('Víctimas - filas duplicadas:', victimas_raw.duplicated().sum())
print('Víctimas - filas completamente vacías:', victimas_raw.isnull().all(axis=1).sum())

Hechos - filas duplicadas: 0 | id_siniestro repetidos: 0


Víctimas - filas duplicadas: 1439
Víctimas - filas completamente vacías: 4


Estadísticas descriptivas de las variables numéricas (se convierten a número solo para describirlas):

In [10]:
numericas = ['numero_total_de_victimas', 'numero_victimas_leve_siniestro', 'numero_victimas_grave_siniestro',
             'numero_victimas_mortal_siniestro', 'mes_siniestro', 'dia_siniestro', 'rango_horario']
hechos_raw[numericas].apply(pd.to_numeric, errors='coerce').describe().round(2)

,numero_total_de_victimas,numero_victimas_leve_siniestro,numero_victimas_grave_siniestro,numero_victimas_mortal_siniestro,mes_siniestro,dia_siniestro,rango_horario
count,62542.00,65818.00,65818.00,65818.00,65818.00,65818.00,65737.00
mean,1.15,1.08,0.05,0.01,6.74,15.61,13.64
std,0.55,0.58,0.23,0.10,3.42,8.75,5.54
min,1.00,0.00,0.00,0.00,1.00,1.00,0.00
25%,1.00,1.00,0.00,0.00,4.00,8.00,10.00
50%,1.00,1.00,0.00,0.00,7.00,15.00,14.00
75%,1.00,1.00,0.00,0.00,10.00,23.00,18.00
max,26.00,26.00,10.00,2.00,12.00,31.00,23.00


Edad de las víctimas y categorías principales:

In [11]:
print(pd.to_numeric(victimas_raw['edad_victima'], errors='coerce').describe().round(1), '\n')
print(hechos_raw['gravedad_siniestro'].value_counts(), '\n')
print(victimas_raw['GRAVEdad_victima'].value_counts())

count    51821.0
mean        37.9
std         15.8
min          0.0
25%         26.0
50%         35.0
75%         47.0
max        116.0
Name: edad_victima, dtype: float64 

gravedad_siniestro
LEVE      62174
GRAVE      2955
MORTAL      689
Name: count, dtype: int64 

GRAVEdad_victima
LEVE      71308
GRAVE      3182
MORTAL      612
mortal       91
Name: count, dtype: int64


Búsqueda de valores problemáticos:

In [12]:
print('Valor "SD" (sin dato) en hora_siniestro:', (hechos_raw['hora_siniestro'] == 'SD').sum())
print('Valor "SD" en numero_total_de_victimas:', (hechos_raw['numero_total_de_victimas'] == 'SD').sum())
print('Celdas con "#¡REF!" (error de Excel):', (hechos_raw == '#¡REF!').sum().sum())
print('Fechas sin cero a la izquierda (ej. 2025-1-29):', (~hechos_raw['fecha_siniestro'].str.match(r'^\d{4}-\d{2}-\d{2}$')).sum())
print('Comunas con formato distinto a "Comuna N":', (~hechos_raw['comuna_siniestro'].fillna('').str.startswith('Comuna')).sum())
print('Contraparte en minúsculas (ej. "auto"):', hechos_raw['contraparte_siniestro'].str.islower().sum())

Valor "SD" (sin dato) en hora_siniestro: 4
Valor "SD" en numero_total_de_victimas: 3276
Celdas con "#¡REF!" (error de Excel): 5
Fechas sin cero a la izquierda (ej. 2025-1-29): 9406
Comunas con formato distinto a "Comuna N": 14824
Contraparte en minúsculas (ej. "auto"): 149


**Hallazgos (siniestros):**

- **Hechos:** 65.818 siniestros de 2019 a 2025, uno por fila, sin identificadores repetidos. El período del trabajo (2021–2025) tiene 49.602.
- **Fechas:** las de 2025 vienen sin ceros a la izquierda (`2025-1-29`); hay que convertirlas con cuidado.
- **Faltantes disfrazados:** el valor `SD` ("sin dato") aparece en la hora (4 casos), en el total de víctimas (3.276, todo 2025) y en categorías; además hay una fila con `#¡REF!`, un error de Excel en la carga original. Pandas no los cuenta como nulos, por eso hay que unificarlos.
- **Comunas:** hasta 2024 dicen `Comuna 1` y en 2025 solo `1`; hay que homogeneizarlas. Las coordenadas están dentro de la Ciudad.
- **Textos** con mayúsculas y minúsculas mezcladas (`auto` y `AUTO`, `mortal` y `MORTAL`).
- **Víctimas:** 4 filas vacías al final del archivo, edades `SD` y una columna mal escrita (`GRAVEdad_victima`). Hay 1.439 filas idénticas, pero **no son errores**: un siniestro puede tener dos víctimas con los mismos datos (por ejemplo, dos acompañantes sin edad registrada).
- Para la pregunta, la variable clave es **fecha y hora** del siniestro: solo 77 hechos no tienen hora.
- **Limitación de la fuente:** solo se registran siniestros con víctimas (lesiones u homicidios culposos); los choques con daños materiales no figuran.

## B. Partidos de River Plate y Boca Juniors

Hay un CSV por equipo y por año (10 archivos) descargados de FBref. Se unen en una tabla para poder explorarlos juntos.

### a) Carga: primeras y últimas filas

In [13]:
RUTA_RB = REPO + 'datos/Temporadas%20River%20-%20Boca/'
archivos = {'Boca Juniors': 'Boca%20Juniors/boca_{}.csv', 'River Plate': 'River%20Plate/river_{}.csv'}
lista = []
for equipo, patron in archivos.items():
    for anio in range(2021, 2026):
        df = pd.read_csv(RUTA_RB + patron.format(anio))
        df.insert(0, 'equipo', equipo)
        df.insert(1, 'temporada', anio)
        lista.append(df)
clubes_raw = pd.concat(lista, ignore_index=True)
clubes_raw.head()

,equipo,temporada,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
0,Boca Juniors,2021,2021-02-14,19:20,Copa de la Liga Profesional,Group stage,Sun,Home,D,2,2,Gimnasia–LP,67.0,NaN,Carlos Tevez,4-2-3-1,4-2-3-1,Pablo Echavarría,Match Report,NaN
1,Boca Juniors,2021,2021-02-21,21:30,Copa de la Liga Profesional,Group stage,Sun,Away,W,1,0,Newell's,53.0,NaN,Carlos Izquierdoz,4-4-2,4-3-3,Darío Herrera,Match Report,NaN
2,Boca Juniors,2021,2021-02-28,21:30,Copa de la Liga Profesional,Group stage,Sun,Home,D,1,1,Sarmiento–J,71.0,NaN,Carlos Izquierdoz,4-4-2,4-2-3-1,Francisco Oscar Lamolina,Match Report,NaN
3,Boca Juniors,2021,2021-03-07,21:30,Copa de la Liga Profesional,Group stage,Sun,Away,W,7,1,Vélez Sarsfield,41.0,NaN,Carlos Tevez,4-1-4-1,4-2-3-1,Diego Abal,Match Report,NaN
4,Boca Juniors,2021,2021-03-14,18:00,Copa de la Liga Profesional,Group stage,Sun,Home,D,1,1,River Plate,NaN,NaN,Carlos Tevez,3-4-1-2,4-3-3,Facundo Tello,Match Report,NaN


Últimas filas:

In [14]:
clubes_raw.tail()

,equipo,temporada,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,Poss,Attendance,Captain,Formation,Opp Formation,Referee,Match Report,Notes
496,River Plate,2025,2025-10-18,20:30,Liga Argentina,Torneo Clausura — Regular season,Sat,Away,W,2,0,Talleres–C,54.0,NaN,Franco Armani,4-3-1-2,4-3-3,Ariel Penel,Match Report,NaN
497,River Plate,2025,2025-11-02,20:30,Liga Argentina,Torneo Clausura — Regular season,Sun,Home,L,0,1,Gimnasia–LP,64.0,NaN,Franco Armani,4-2-3-1,4-2-3-1,Nazareno Arasa,Match Report,NaN
498,River Plate,2025,2025-11-09,16:30,Liga Argentina,Torneo Clausura — Regular season,Sun,Away,L,0,2,Boca Juniors,55.0,NaN,Franco Armani,3-5-2,4-2-2-2,Nicolás Ramírez,Match Report,NaN
499,River Plate,2025,2025-11-16,17:00,Liga Argentina,Torneo Clausura — Regular season,Sun,Away,D,0,0,Vélez Sarsfield,55.0,NaN,Enzo Pérez,4-1-3-2,4-2-3-1,Leandro Rey,Match Report,NaN
500,River Plate,2025,2025-11-24,19:15,Liga Argentina,Torneo Clausura — Round of 16,Mon,Away,L,2,3,Racing Club,47.0,NaN,Enzo Pérez,4-3-1-2,4-3-3,Facundo Tello,Match Report,NaN


### b) Estructura y cobertura

In [15]:
print(clubes_raw.shape[0], 'partidos x', clubes_raw.shape[1], 'columnas\n')
print(clubes_raw.dtypes)

501 partidos x 20 columnas

equipo               str
temporada          int64
Date                 str
Time                 str
Comp                 str
Round                str
Day                  str
Venue                str
Result               str
GF                object
GA                object
Opponent             str
Poss             float64
Attendance       float64
Captain              str
Formation            str
Opp Formation        str
Referee              str
Match Report         str
Notes                str
dtype: object


Cobertura temporal y por competencia:

In [16]:
print(pd.crosstab(clubes_raw['temporada'], clubes_raw['equipo'], margins=True), '\n')
print(clubes_raw['Comp'].value_counts(), '\n')
print(clubes_raw['Venue'].value_counts())

equipo     Boca Juniors  River Plate  All
temporada                                
2021                 48           49   97
2022                 52           50  102
2023                 54           51  105
2024                 53           54  107
2025                 42           48   90
All                 249          252  501 

Comp
Liga Argentina                 284
Copa de la Liga Profesional    122
Libertadores                    79
Sudamericana                    10
Club World Cup                   6
Name: count, dtype: int64 

Venue
Home       253
Away       242
Neutral      6
Name: count, dtype: int64


### c) Calidad: nulos, duplicados y estadísticas descriptivas

In [17]:
print(clubes_raw.isnull().sum(), '\n')
print('Filas duplicadas:', clubes_raw.duplicated().sum(),
      '| Partidos repetidos (equipo + fecha):', clubes_raw.duplicated(['equipo', 'Date']).sum())

equipo             0
temporada          0
Date               0
Time               0
Comp               0
Round              0
Day                0
Venue              0
Result             0
GF                 0
GA                 0
Opponent           0
Poss               2
Attendance       466
Captain            0
Formation          0
Opp Formation      1
Referee           40
Match Report       0
Notes            458
dtype: int64 

Filas duplicadas: 0 | Partidos repetidos (equipo + fecha): 0


Estadísticas descriptivas de las variables numéricas (posesión y público) y del resultado según FBref:

In [18]:
print(clubes_raw[['Poss', 'Attendance']].describe().round(1), '\n')
print(clubes_raw.groupby('equipo')['Result'].value_counts().unstack())

        Poss  Attendance
count  499.0        35.0
mean    59.5     43556.5
std     10.6     17687.7
min     25.0      8083.0
25%     53.0     33874.5
50%     61.0     44603.0
75%     67.0     59585.0
max     84.0     69232.0 

Result         D   L    W
equipo                   
Boca Juniors  74  60  115
River Plate   69  54  129


Búsqueda de valores problemáticos:

In [19]:
con_parentesis = clubes_raw['Time'].str.contains(r'\(')
print('Horarios con paréntesis:', con_parentesis.sum(), '->', clubes_raw.loc[con_parentesis, 'Time'].unique()[:4].tolist())
goles_raros = clubes_raw['GF'].astype(str).str.contains(r'\(')
print('Goles con paréntesis:', goles_raros.sum(), '->', clubes_raw.loc[goles_raros, 'GF'].unique()[:4].tolist())
print('Notas sobre penales:', clubes_raw['Notes'].fillna('').str.contains('penalty').sum())
print('Rivales con código de país:', clubes_raw['Opponent'].str.match(r'^[a-z]{2} ').sum(),
      '->', clubes_raw.loc[clubes_raw['Opponent'].str.match(r'^[a-z]{2} '), 'Opponent'].unique()[:3].tolist())

Horarios con paréntesis: 29 -> ['18:00 (19:00)', '19:30 (21:30)', '20:00 (21:00)', '19:00 (21:00)']
Goles con paréntesis: 17 -> ['1 (4)', '0 (2)', '0 (1)', '0 (6)']
Notas sobre penales: 17
Rivales con código de país: 95 -> ['bo The Strongest', 'br Santos', 'ec Barcelona']


**Hallazgos (River y Boca):**

- **501 partidos** (249 de Boca y 252 de River), unos 50 por equipo y por año, sin duplicados.
- **Horario:** FBref informa la hora local del estadio; cuando el partido fue en otro huso horario agrega la **hora argentina entre paréntesis** (ej. `19:30 (21:30)` en Ecuador). Hay que usar siempre la hora argentina.
- **Penales:** en 17 partidos los goles vienen como texto (`1 (4)`: goles y penales), por eso `GF` y `GA` no son numéricos. FBref los marca como empate (`D`) aunque un equipo haya quedado eliminado.
- Los rivales internacionales llevan un código de país adelante (`br Palmeiras`).
- `Attendance` y `Notes` tienen muchos nulos; solo `Notes` sirve (indica alargue y penales).
- **Cobertura:** Liga, Copa de la Liga, Libertadores, Sudamericana y Mundial de Clubes. **No figuran la Copa Argentina ni las supercopas**, que se informan como limitación.

## C. Partidos de la Selección Argentina

La base de resultados internacionales tiene todos los partidos entre selecciones desde 1872. Para explorarla en el alcance del trabajo, se filtran los partidos de Argentina entre 2021 y 2025.

### a) Carga: primeras y últimas filas

In [20]:
RUTA_INT = REPO + 'datos/Resultados%20Internacionales%201872%20-%202026/'
internacionales = pd.read_csv(RUTA_INT + 'results.csv', parse_dates=['date'])
print('Base completa:', internacionales.shape)
es_arg = (internacionales['home_team'] == 'Argentina') | (internacionales['away_team'] == 'Argentina')
arg_raw = internacionales[es_arg & internacionales['date'].between('2021-01-01', '2025-12-31')].reset_index(drop=True)
arg_raw.head()

Base completa: (49945, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2021-06-03,Argentina,Chile,1,1,FIFA World Cup qualification,Santiago del Estero,Argentina,False
1,2021-06-08,Colombia,Argentina,2,2,FIFA World Cup qualification,Barranquilla,Colombia,False
2,2021-06-14,Argentina,Chile,1,1,Copa América,Rio de Janeiro,Brazil,True
3,2021-06-18,Argentina,Uruguay,1,0,Copa América,Brasília,Brazil,True
4,2021-06-21,Argentina,Paraguay,1,0,Copa América,Brasília,Brazil,True


Últimas filas:

In [21]:
arg_raw.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
62,2025-09-04,Argentina,Venezuela,3,0,FIFA World Cup qualification,Buenos Aires,Argentina,False
63,2025-09-09,Ecuador,Argentina,1,0,FIFA World Cup qualification,Guayaquil,Ecuador,False
64,2025-10-10,Argentina,Venezuela,1,0,Friendly,Miami Gardens,United States,True
65,2025-10-14,Puerto Rico,Argentina,0,6,Friendly,Fort Lauderdale,United States,True
66,2025-11-14,Angola,Argentina,0,2,Friendly,Luanda,Angola,False


### b) Estructura y cobertura

In [22]:
print(arg_raw.shape[0], 'partidos x', arg_raw.shape[1], 'columnas\n')
print(arg_raw.dtypes, '\n')
print(arg_raw['tournament'].value_counts())

67 partidos x 9 columnas

date          datetime64[us]
home_team                str
away_team                str
home_score             int64
away_score             int64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object 

tournament
FIFA World Cup qualification      31
Friendly                          15
Copa América                      13
FIFA World Cup                     7
CONMEBOL–UEFA Cup of Champions     1
Name: count, dtype: int64


### c) Calidad: nulos, duplicados y estadísticas descriptivas

In [23]:
print(arg_raw.isnull().sum(), '\n')
print('Duplicados:', arg_raw.duplicated().sum())
arg_raw[['home_score', 'away_score']].describe()

date          0
home_team     0
away_team     0
home_score    0
away_score    0
tournament    0
city          0
country       0
neutral       0
dtype: int64 

Duplicados: 0


,home_score,away_score
count,67.000000,67.000000
mean,1.626866,0.910448
std,1.525896,1.215198
min,0.000000,0.000000
25%,0.500000,0.000000
50%,1.000000,1.000000
75%,3.000000,1.000000
max,7.000000,6.000000


Columnas disponibles y tandas de penales registradas en el archivo complementario `shootouts.csv`:

In [24]:
print('Columnas:', arg_raw.columns.tolist())
print('¿Tiene hora del partido?:', any('time' in c.lower() or 'hora' in c.lower() for c in arg_raw.columns))
penales_raw = pd.read_csv(RUTA_INT + 'shootouts.csv', parse_dates=['date'])
pen_arg = penales_raw[((penales_raw['home_team'] == 'Argentina') | (penales_raw['away_team'] == 'Argentina'))
                      & penales_raw['date'].between('2021-01-01', '2025-12-31')]
pen_arg

Columnas: ['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament', 'city', 'country', 'neutral']
¿Tiene hora del partido?: False


,date,home_team,away_team,winner,first_shooter
582,2021-07-06,Argentina,Colombia,Argentina,Colombia
615,2022-12-09,Netherlands,Argentina,Argentina,Netherlands
616,2022-12-18,Argentina,France,Argentina,France
652,2024-07-04,Argentina,Ecuador,Argentina,Argentina


**Hallazgos (Selección):**

- **67 partidos** (31 de Eliminatorias, 15 amistosos, 13 de Copa América, 7 del Mundial y la Finalissima), sin nulos ni duplicados. El partido suspendido contra Brasil (5/9/2021) no figura, lo cual es correcto.
- **Problema principal: la base no tiene la hora de los partidos**, imprescindible para la ventana de 12 h. Hay que completarla con otra fuente.
- Los goles están como local/visitante, no desde el punto de vista de Argentina.
- Los 4 partidos definidos por penales figuran como empate; `shootouts.csv` dice quién ganó la tanda.

## Resumen de la exploración

| Problema | Base | Se resuelve en 3.3 con… |
|---|---|---|
| Faltantes como `SD` y `#¡REF!` | Siniestros | Unificarlos como nulos |
| Fechas, comunas y textos con formatos mezclados | Siniestros | Convertir tipos y normalizar |
| Años fuera del período | Siniestros | Filtrar 2021–2025 |
| Fecha y hora en columnas separadas | Siniestros | Crear `fecha_hora` |
| Hora local del estadio | River/Boca | Usar la hora argentina |
| Penales como empate y goles como texto | River/Boca y Selección | Separar goles y penales; reclasificar el resultado |
| Falta la hora del partido | Selección | Completarla con Wikipedia y pasarla a hora argentina |